Randomly load volumes into training/validation

Note that each volume is of a different human, and we have 20 volumes

Split: 16 train & 4 validation

In [11]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [12]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
import nibabel as nib
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

training_path = Path('/home/thomas/Downloads/heartDisease/imagesTr')
labels_path = Path('/home/thomas/Downloads/heartDisease/labelsTr')
test_path = Path('/home/thomas/Downloads/heartDisease/imagesTs')

# Get sorted image and label paths
image_paths = sorted(list(training_path.glob('*.nii.gz')))
label_paths = sorted(list(labels_path.glob('*.nii.gz')))

# Check same number
assert len(image_paths) == len(label_paths), "Number of images and labels does not match"

# Check filenames match
for img, lbl in zip(image_paths, label_paths):
    assert img.name == lbl.name, f"Mismatch: {img.name} vs {lbl.name}"

# Pair images and labels together
pairs = list(zip(image_paths, label_paths))

# Split pairs together
train_pairs, val_pairs = train_test_split(
    pairs,
    test_size=0.2,
    random_state=42
)
# Unzip pairs back into separate lists
train_images, train_labels = zip(*train_pairs)
val_images, val_labels = zip(*val_pairs)

# Convert to lists of strings
train_images = [str(p) for p in train_images]
train_labels = [str(p) for p in train_labels]
val_images = [str(p) for p in val_images]
val_labels = [str(p) for p in val_labels]
test_images = [str(p) for p in sorted(test_path.glob('*.nii.gz'))]

print(f"Training images: {len(train_images)}")
print(f"Validation images: {len(val_images)}")
print(f"Training labels: {len(train_labels)}")
print(f"Validation labels: {len(val_labels)}")
print(f"Test images: {len(test_images)}")

Training images: 16
Validation images: 4
Training labels: 16
Validation labels: 4
Test images: 10


Create a dataset class

In [ ]:
class HeartDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, label_paths=None):
        self.image_paths = image_paths
        self.label_paths = label_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = nib.load(self.image_paths[idx]).get_fdata().astype(np.float32)

        # basic preprocessing
        image = (image - image.mean()) / (image.std() + 1e-8)

        # add channel dimension: [D, H, W] -> [1, D, H, W] (the form pytorch expects)
        image = torch.from_numpy(image).unsqueeze(0)

        if self.label_paths is not None: #if labels are provided, load them
            label = nib.load(self.label_paths[idx]).get_fdata().astype(np.int64)
            label = torch.from_numpy(label)

            return image, label

        return image

create a dataloader

In [15]:
from torch.utils.data import DataLoader
train_dataset = HeartDataset(train_images, train_labels)
val_dataset = HeartDataset(val_images, val_labels)
test_dataset = HeartDataset(test_images)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

Define model:

In [16]:
import torch.nn as nn

class Simple3DUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv3d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv3d(16, 16, 3, padding=1),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Conv3d(16, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv3d(16, 2, 1)  # 2 classes: background + atrium
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

Define loss

In [17]:
loss_fn = nn.CrossEntropyLoss()


Define optimizer

In [18]:
model = Simple3DUNet()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

Training loop

In [20]:
num_epochs = 1

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        outputs = model(images)

        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch}, Loss: {total_loss:.4f}")


Epoch 0, Loss: 9.7545


validation loop

In [21]:
model.eval()

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)

        preds = torch.argmax(outputs, dim=1)

        # compute dice (you’ll add this)